# The adaptive cut with scikit-learn

`ac.AdaptiveClustering` works like `sklearn.cluster.AgglomerativeClustering` (same `linkage` and
`metric` parameters, `labels_`, `children_`, `distances_`), but it needs no number of clusters
and no distance threshold. It passes scikit-learn's estimator checks, so it works with
`clone`, pipelines and `set_params`.

This notebook needs `pip install adaptive_cut scikit-learn`.

In [1]:
import numpy as np
from sklearn.base import clone
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import load_digits, load_iris, load_wine, make_blobs
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score as ari
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import adaptive_cut as ac

## Fit and predict

In [2]:
X, y = make_blobs(600, centers=5, cluster_std=[0.5, 0.5, 1.0, 2.0, 2.5], random_state=3)

model = ac.AdaptiveClustering(linkage="ward", random_state=0).fit(X)
print("clusters:", model.n_clusters_, "| ARI", round(ari(y, model.labels_), 2))
print("tree:", model.children_.shape, model.distances_[:3])
print("sampler:", model.result_)

clusters: 4 | ARI 0.78
tree: (599, 2) [0.00341451 0.00836898 0.00860565]
sampler: AdaptiveCutResult(value=1105.03, n_clusters=4, initial=1105.03, n_steps=10000, T=1.77, engine='numba')


## In a pipeline, next to `AgglomerativeClustering`

`AgglomerativeClustering` needs the number of clusters. The adaptive cut chooses it from the
objective.

In [3]:
data = {
    "iris": load_iris(return_X_y=True),
    "wine": load_wine(return_X_y=True),
    "digits": load_digits(return_X_y=True),
}
for name, (X, y) in data.items():
    prep = PCA(10, random_state=0) if name == "digits" else StandardScaler()
    k = len(np.unique(y))
    agg = make_pipeline(clone(prep), AgglomerativeClustering(n_clusters=k)).fit_predict(X)
    ada = make_pipeline(clone(prep), ac.AdaptiveClustering(random_state=0)).fit_predict(X)
    print(
        f"{name:>7}: AgglomerativeClustering(k={k:2d}) ARI {ari(y, agg):.2f} | "
        f"AdaptiveClustering {len(np.unique(ada)):2d} clusters ARI {ari(y, ada):.2f}"
    )

   iris: AgglomerativeClustering(k= 3) ARI 0.62 | AdaptiveClustering  5 clusters ARI 0.57
   wine: AgglomerativeClustering(k= 3) ARI 0.79 | AdaptiveClustering  3 clusters ARI 0.79


 digits: AgglomerativeClustering(k=10) ARI 0.70 | AdaptiveClustering 26 clusters ARI 0.58


## Trying settings with `set_params`

The estimator follows scikit-learn's parameter conventions.

In [4]:
X, y = load_wine(return_X_y=True)
pipe = make_pipeline(StandardScaler(), ac.AdaptiveClustering(random_state=0))
for linkage in ["ward", "average", "complete"]:
    for covariance in ["diag", "spherical"]:
        pipe.set_params(
            adaptiveclustering__linkage=linkage, adaptiveclustering__covariance=covariance
        )
        labels = pipe.fit_predict(X)
        print(
            f"{linkage:>8} {covariance:>9}: {len(np.unique(labels))} clusters, ARI {ari(y, labels):.2f}"
        )

    ward      diag: 3 clusters, ARI 0.79


    ward spherical: 12 clusters, ARI 0.52
 average      diag: 7 clusters, ARI 0.81
 average spherical: 15 clusters, ARI 0.79
complete      diag: 5 clusters, ARI 0.68
complete spherical: 16 clusters, ARI 0.65


## A tree from `AgglomerativeClustering`

`ac.linkage_from_sklearn` turns a fitted `AgglomerativeClustering` (fitted with
`distance_threshold=0, n_clusters=None`, so the tree is complete and has its heights) into a
scipy linkage matrix.

In [5]:
X, y = make_blobs(600, centers=5, cluster_std=[0.5, 0.5, 1.0, 2.0, 2.5], random_state=3)
tree = AgglomerativeClustering(linkage="average", distance_threshold=0, n_clusters=None).fit(X)
Z = ac.linkage_from_sklearn(tree)
res = ac.adaptive_cut(Z, ac.gaussian_scores(Z, X), T="auto", rng=0)
print(res.best, f"ARI {ari(y, res.labels):.2f}")

Cut(n_clusters=4, value=996.514, height=6.20995) ARI 0.73


## Your own objective in the estimator

`objective` takes a function `objective(Z, X)` returning one score per tree node.

In [6]:
def neg_sse_with_penalty(Z, X, penalty=100.0):
    def score(idx):
        P = X[idx]
        return -((P - P.mean(axis=0)) ** 2).sum() - penalty

    return ac.node_scores(Z, score)


model = ac.AdaptiveClustering(objective=neg_sse_with_penalty, random_state=0).fit(X)
print(model.n_clusters_, "clusters, ARI", round(ari(y, model.labels_), 2))

10 clusters, ARI 0.75
